In [3]:
!pip install torch --index-url https://download.pytorch.org/whl/cpu

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
     |████████████████████████████████| 184.0 MB 25 kB/s              
     |████████████████████████████████| 200 kB 4.9 MB/s            
     |████████████████████████████████| 6.3 MB 38.2 MB/s            
     |████████████████████████████████| 536 kB 71.3 MB/s            


In [4]:
!pip install datarec-lib boto3 requests pandas sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 488 kB 4.6 MB/s            
     |████████████████████████████████| 12.0 MB 78.0 MB/s            
     |████████████████████████████████| 6.6 MB 76.2 MB/s            
     |████████████████████████████████| 625 kB 75.1 MB/s            
     |████████████████████████████████| 4.2 MB 77.1 MB/s            
     |████████████████████████████████| 56 kB 9.7 MB/s             
     |████████████████████████████████| 791 kB 78.9 MB/s            
     |████████████████████████████████| 566 kB 72.6 MB/s            
     |████████████████████████████████| 3.3 MB 59.2 MB/s            
     |████████████████████████████████| 507 kB 59.9 MB/s            
     |████████████████████████████████| 310 kB 78.7 MB/s            
     |████████████████████████████████| 98 kB 15.7 MB/s            
     |████████████████████████████████| 87 kB 13.8 MB/s            


In [8]:
# imports
import boto3
import requests
import os
from botocore.exceptions import ClientError
import zipfile
import pandas as pd
import pickle
from sentence_transformers import SentenceTransformer, util
import json
import random
import datetime
import torch

In [2]:
# This function downloads the checks to see if the dataset exists in s3 bucket, downloads it if not, and cleans up the zip file
def download_dataset(BUCKET_NAME, S3_FILE_KEY, DATASET_URL, LOCAL_FILE_NAME):
    s3 = boto3.client('s3')
    
    # Check if it already exists in s3
    file_exists = False
    try:
        s3.head_object(Bucket=BUCKET_NAME, Key=S3_FILE_KEY)
        file_exists = True
        print(f"Dataset already exists in S3 bucket '{BUCKET_NAME}'")
    except ClientError as e:
        if e.response['Error']['Code'] == '404':
            print("Dataset not in S3... starting download")
        else:
            print(f"Error accessing S3: {e}")
    
    # Download if it doesn't exist
    if not file_exists:
        print("Downloading dataset")
        response = requests.get(DATASET_URL, stream=True)
        response.raise_for_status() 
        
        with open(LOCAL_FILE_NAME, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
                
        print("Download complete. Uploading to S3...")
        s3.upload_file(LOCAL_FILE_NAME, BUCKET_NAME, S3_FILE_KEY)
        print("Upload successful")
        
        # Clean up the local zip file on the EC2 instance to save space
        if os.path.exists(LOCAL_FILE_NAME):
            os.remove(LOCAL_FILE_NAME)
            print("Deleted zip file from EC2 to save space")

In [16]:
# This function downloads the data from the s3 bucket, 
# parses and filters the data, creates BERT embeddings, 
# and uploads them to the s3 bucket


def create_BERT_embeddings(S3_ZIP_KEY, LOCAL_ZIP, EXTRACT_DIR, S3_OUTPUT_KEY, LOCAL_OUTPUT, YEAR_FILTER=9999):
    s3 = boto3.client('s3')
    
    # Fetch and Extract the Data
    print("Fetching dataset from S3...")
    s3.download_file(BUCKET_NAME, S3_ZIP_KEY, LOCAL_ZIP)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    
    # Parse the Movies Dataset
    print("Loading movies.dat...")
    # MovieLens uses '::' as a separator and requires latin-1 encoding
    movies_path = os.path.join(EXTRACT_DIR, 'ml-1m', 'movies.dat')
    movies = pd.read_csv(
        movies_path, 
        sep='::', 
        engine='python', 
        encoding='latin-1', 
        names=['MovieID', 'Title', 'Genres']
    )
    
    # Filter for Year
    if YEAR_FILTER != 9999:
        print(f"Filtering for movies released in or before {YEAR_FILTER}...")
        # Extract the 4 digits inside the parentheses at the end of the title
        movies['Year'] = movies['Title'].str.extract(r'\((\d{4})\)').astype(float)
        movies_filtered = movies[movies['Year'] <= YEAR_FILTER].copy()
    else:
        print(f"Not filtering for year... all movies will be included")
        movies_filtered = movies.copy()
    
    print(f"Filtered dataset down to {len(movies_filtered)} movies.")
    
    # Prepare Text for BERT
    # We combine the title and genres so BERT understands the full context
    movies_filtered['bert_text'] = movies_filtered['Title'] + " Genres: " + movies_filtered['Genres'].str.replace('|', ' ', regex=False)
    
    # Generate BERT Embeddings
    print("Loading BERT model (this may take a moment to download weights)...")
    # 'all-MiniLM-L6-v2' is a fast, highly efficient standard BERT model for embeddings
    model = SentenceTransformer('all-MiniLM-L6-v2') 
    
    print("Generating embeddings (this is the heavy computation step)...")
    # Convert the text column to a list and encode
    texts_to_embed = movies_filtered['bert_text'].tolist()
    embeddings = model.encode(texts_to_embed, show_progress_bar=True)
    
    # Save Intermediate Results to a Pickle File
    print("Saving intermediate embeddings...")
    output_data = {
        'MovieID': movies_filtered['MovieID'].tolist(),
        'Title': movies_filtered['Title'].tolist(),
        'Embeddings': embeddings
    }
    
    with open(LOCAL_OUTPUT, 'wb') as f:
        pickle.dump(output_data, f)
    
    # Upload to S3
    print("Uploading embeddings to S3...")
    s3.upload_file(LOCAL_OUTPUT, BUCKET_NAME, S3_OUTPUT_KEY)
    print(f"Embeddings saved to S3 as '{S3_OUTPUT_KEY}'.")
    
    # Clean up local files
    os.remove(LOCAL_ZIP)
    os.remove(LOCAL_OUTPUT)

In [68]:
# This function will generate 5 movie recommendations for different types of users

def generate_recommendations(BUCKET_NAME, S3_EMBEDDINGS_KEY, EXTRACT_DIR, OUTPUT_FILE_NAME, PERSONAL=False, MY_RATINGS=None):
    s3 = boto3.client('s3')
    
    # Download the desired BERT embeddings
    print(f"Loading BERT embeddings from {S3_EMBEDDINGS_KEY}...")
    local_emb_file = S3_EMBEDDINGS_KEY 
    if not os.path.exists(local_emb_file):
        s3.download_file(BUCKET_NAME, S3_EMBEDDINGS_KEY, local_emb_file) # download the pickle file locally from s3
        
    with open(local_emb_file, 'rb') as f:
        emb_data = pickle.load(f) # open the local file to work with the embeddings
        
    emb_movie_ids = emb_data['MovieID'] # parse the data to work with IDs and Titles
    emb_titles = emb_data['Title']
    embeddings = torch.tensor(emb_data['Embeddings']) # Convert the arrays in the pkl file into PyTorch tensor so we can do optimized ML 
                                                      # on the embeddings later
    
    # Give all movie IDs their own idx, makes for easy searching across tensors
    id_to_idx = {m_id: i for i, m_id in enumerate(emb_movie_ids)}
    
    # Load the Ratings and Users datasets
    print("Loading Ratings and Users datasets...")
    ratings = pd.read_csv(os.path.join(EXTRACT_DIR, 'ml-1m', 'ratings.dat'), sep='::', engine='python', names=['UserID', 'MovieID', 'Rating', 'Timestamp'])
    users = pd.read_csv(os.path.join(EXTRACT_DIR, 'ml-1m', 'users.dat'), sep='::', engine='python', names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'])
    
    # Filter ratings to only look at the movies we have embeddings for (such as filtered for pre-1980)
    valid_movie_ids = set(emb_movie_ids)
    valid_ratings = ratings[ratings['MovieID'].isin(valid_movie_ids)]
    
    output_results = []

    # IF PERSONAL
    if PERSONAL and MY_RATINGS is not None:
        print("Generating Personal recommendations...")
        
        # Get personal ratings
        my_favorites = [m_id for m_id, rating in MY_RATINGS.items() if rating >= 4]
        
        fav_indices = [id_to_idx[m_id] for m_id in my_favorites if m_id in id_to_idx]
        fav_embeddings = embeddings[fav_indices]
        my_profile_embedding = torch.mean(fav_embeddings, dim=0, keepdim=True)
        
        # Find unseen movies
        seen_movie_ids = set(MY_RATINGS.keys())
        unseen_indices = [i for i, m_id in enumerate(emb_movie_ids) if m_id not in seen_movie_ids]
        unseen_embeddings = embeddings[unseen_indices]
        
        # Calculate cosine similarity and get top 5
        cos_scores = util.cos_sim(my_profile_embedding, unseen_embeddings)[0]
        top_5_indices = torch.topk(cos_scores, k=5).indices.tolist()
        top_5_titles = [emb_titles[unseen_indices[i]] for i in top_5_indices]
        
        # 4. Format output
        my_profile_details = [
            {"MovieID": m_id, "Title": emb_titles[id_to_idx[m_id]], "Rating": rating}
            for m_id, rating in MY_RATINGS.items() if m_id in id_to_idx
        ]
        
        output_results.append({
            "User_Type": "Personal Profile",
            "My_Ratings": my_profile_details,
            "Recommended_Movies": top_5_titles
        })
        
    # IF NOT PERSONAL
    else:
        
        # -------------------------------------------------------------------------------
        # Generate movie recs for Cold Users
        # Our strategy here is just to give them the most popular and highly rated movies
        # -------------------------------------------------------------------------------
        print("Generating Cold User recommendations...")
        
        # Create new db of movie IDs with their rating mean and counts as data
        movie_stats = valid_ratings.groupby('MovieID').agg(
            rating_mean=('Rating', 'mean'),
            rating_count=('Rating', 'count')
        ).reset_index() # Reset index so we can search and sort by index rather than ID, much easier
        
        # Filter for movies with at least 50 ratings (avoid one offs)
        popular_movies = movie_stats[movie_stats['rating_count'] >= 50].sort_values(by='rating_mean', ascending=False)
        top_5_cold_ids = popular_movies.head(5)['MovieID'].tolist() # get the top 5 movies
        top_5_cold_titles = [emb_titles[id_to_idx[m_id]] for m_id in top_5_cold_ids]
        
        cold_user_data = {
            "User_Type": "Cold User",
            "Last_Interaction_Time": "N/A",
            "User_Summary": {"Gender": "N/A", "Age": "N/A", "Occupation": "N/A"},
            "Recommended_Movies": top_5_cold_titles
        }
        output_results.append(cold_user_data)
        
        # ------------------------------------------------------------------------------------
        # Generate movie recs for Top Users
        # Uses ML (cosine similarity, embeddings) to find movies similar to the ones they like
        # ------------------------------------------------------------------------------------
        print("Generating Top User recommendations...")
        
        # Find users in the top 5% of interaction counts
        user_counts = ratings['UserID'].value_counts() # get num ratings of each user
        threshold = user_counts.quantile(0.95) # set the threshold to top 5% of users
        top_users = user_counts[user_counts >= threshold].index.tolist() # filter
        
        # Pick a random top user who has highly rated at least movie (>= 4 stars) within our desired time frame
        # The reason we need to wrap it in a while loop is in case we pick a top user who hasn't
        # rated a movie made before our filter date (1980)
        top_user_id = None
        user_favorites = []
        while not user_favorites: 
            top_user_id = random.choice(top_users)
            user_ratings = valid_ratings[(valid_ratings['UserID'] == top_user_id) & (valid_ratings['Rating'] >= 4)]
            user_favorites = user_ratings['MovieID'].tolist()
            
        # Gather user metadata for the output
        user_info = users[users['UserID'] == top_user_id].iloc[0]
        user_all_ratings = ratings[ratings['UserID'] == top_user_id]
        
        # Get Last_Interaction_Time, need to convert fomr Unix to readable datetime
        last_timestamp = user_all_ratings['Timestamp'].max()
        last_interaction = datetime.datetime.fromtimestamp(last_timestamp).strftime('%Y-%m-%d %H:%M:%S')
        
        # -----------------
        # COSINE SIMILARITY
        # -----------------
        
        # Average all filtered favorite movies into one embedding (kind of like the "root" embedding)
        fav_indices = [id_to_idx[m_id] for m_id in user_favorites]
        fav_embeddings = embeddings[fav_indices]
        user_mean_embedding = torch.mean(fav_embeddings, dim=0, keepdim=True)
        
        # Find all filtered movies that the user hasn't seen
        seen_movie_ids = set(user_all_ratings['MovieID'].tolist())
        unseen_indices = [i for i, movie_id in enumerate(emb_movie_ids) if movie_id not in seen_movie_ids]
        unseen_embeddings = embeddings[unseen_indices]
        
        # Calculate Cosine Similarity between the user's "root" embedding and the unseen movies embeddings
        cos_scores = util.cos_sim(user_mean_embedding, unseen_embeddings)[0]
        
        # Get the top 5 movies that are similar
        top_5_indices = torch.topk(cos_scores, k=5).indices.tolist()
        top_5_topuser_titles = [emb_titles[unseen_indices[i]] for i in top_5_indices]
        
        top_user_data = {
            "User_Type": "Top User",
            "UserID_Reference": int(top_user_id),
            "Last_Interaction_Time": last_interaction,
            "User_Summary": {
                "Gender": user_info['Gender'],
                "Age_Category": int(user_info['Age']),
                "Occupation_Category": int(user_info['Occupation'])
            },
            "Recommended_Movies": top_5_topuser_titles
        }
        output_results.append(top_user_data)
    
    # Save the recs and upload to s3 
    with open(OUTPUT_FILE_NAME, 'w') as f:
        json.dump(output_results, f, indent=4)
        
    print(f"Uploading {OUTPUT_FILE_NAME} to S3...")
    s3.upload_file(OUTPUT_FILE_NAME, BUCKET_NAME, OUTPUT_FILE_NAME)
    print(f"Recommendations saved to S3 as '{OUTPUT_FILE_NAME}'.")

    

In [69]:
# Show what movies are recommended
def print_recommnedations(FILE_PATH):
    # Open and load the JSON file
    with open(FILE_PATH, 'r') as f:
        results = json.load(f)
    
    # Loop through and print the recommendations
    for user_data in results:
        print(f"--- {user_data['User_Type']} ---")
        
        # Print the User ID if it's the Top User
        if 'UserID_Reference' in user_data:
            print(f"User ID: {user_data['UserID_Reference']}")
            
        print("Recommended Movies:")
        for i, movie in enumerate(user_data['Recommended_Movies'], start=1):
            print(f"  {i}. {movie}")
        
        print("\n")

In [70]:
# Part 1
BUCKET_NAME = 'yi-alex-hw2'
S3_FILE_KEY = 'ml-1m.zip' 
DATASET_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
LOCAL_FILE_NAME = 'ml-1m.zip'

download_dataset(BUCKET_NAME, S3_FILE_KEY, DATASET_URL, LOCAL_FILE_NAME)

Dataset already exists in S3 bucket 'yi-alex-hw2'


/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [18]:
# Part 2
S3_ZIP_KEY = 'ml-1m.zip'
LOCAL_ZIP = 'ml-1m.zip'
EXTRACT_DIR = 'ml-1m-data'
S3_OUTPUT_KEY = 'bert_embeddings_pre1980.pkl'
LOCAL_OUTPUT = 'bert_embeddings_pre1980.pkl'
YEAR_FILTER = 1980

create_BERT_embeddings(S3_ZIP_KEY, LOCAL_ZIP, EXTRACT_DIR, S3_OUTPUT_KEY, LOCAL_OUTPUT, YEAR_FILTER)

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Fetching dataset from S3...
Extracting dataset...
Loading movies.dat...
Filtering for movies released in or before 1980...
Filtered dataset down to 887 movies.
Loading BERT model (this may take a moment to download weights)...
Generating embeddings (this is the heavy computation step)...


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Saving intermediate embeddings...
Uploading embeddings to S3...
Embeddings saved to S3 as 'bert_embeddings_pre1980.pkl'.


In [71]:
# Part 3
BUCKET_NAME = 'yi-alex-hw2'
S3_EMBEDDINGS_KEY = 'bert_embeddings_pre1980.pkl'
EXTRACT_DIR = 'ml-1m-data'
OUTPUT_FILE_NAME = 'user_recommendations.json'

generate_recommendations(BUCKET_NAME, S3_EMBEDDINGS_KEY, EXTRACT_DIR, OUTPUT_FILE_NAME)

print_recommnedations('user_recommendations.json')

Loading BERT embeddings from bert_embeddings_pre1980.pkl...
Loading Ratings and Users datasets...
Generating Cold User recommendations...
Generating Top User recommendations...
Uploading user_recommendations.json to S3...
Recommendations saved to S3 as 'user_recommendations.json'.
--- Cold User ---
Recommended Movies:
  1. Sanjuro (1962)
  2. Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)
  3. Godfather, The (1972)
  4. Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)
  5. Rear Window (1954)


--- Top User ---
User ID: 5111
Recommended Movies:
  1. Windows (1980)
  2. Very Natural Thing, A (1974)
  3. 8 1/2 (1963)
  4. Our Town (1940)
  5. Limelight (1952)




In [72]:
# Part 4
S3_ZIP_KEY = 'ml-1m.zip'
LOCAL_ZIP = 'ml-1m.zip'
EXTRACT_DIR = 'ml-1m-data'
S3_OUTPUT_KEY = 'bert_embeddings_fulldata.pkl'
LOCAL_OUTPUT = 'bert_embeddings_fulldata.pkl'
YEAR_FILTER = 9999

create_BERT_embeddings(S3_ZIP_KEY, LOCAL_ZIP, EXTRACT_DIR, S3_OUTPUT_KEY, LOCAL_OUTPUT, YEAR_FILTER)

BUCKET_NAME = 'yi-alex-hw2'
S3_EMBEDDINGS_KEY = 'bert_embeddings_fulldata.pkl'
EXTRACT_DIR = 'ml-1m-data'
OUTPUT_FILE_NAME = 'user_recommendations_full.json'

generate_recommendations(BUCKET_NAME, S3_EMBEDDINGS_KEY, EXTRACT_DIR, OUTPUT_FILE_NAME)

print_recommnedations('user_recommendations_full.json')

Fetching dataset from S3...


/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Extracting dataset...
Loading movies.dat...
Not filtering for year... all movies will be included
Filtered dataset down to 3883 movies.
Loading BERT model (this may take a moment to download weights)...
Generating embeddings (this is the heavy computation step)...


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

Saving intermediate embeddings...
Uploading embeddings to S3...
Embeddings saved to S3 as 'bert_embeddings_fulldata.pkl'.
Loading BERT embeddings from bert_embeddings_fulldata.pkl...
Loading Ratings and Users datasets...


/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Generating Cold User recommendations...
Generating Top User recommendations...
Uploading user_recommendations_full.json to S3...
Recommendations saved to S3 as 'user_recommendations_full.json'.
--- Cold User ---
Recommended Movies:
  1. Sanjuro (1962)
  2. Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)
  3. Shawshank Redemption, The (1994)
  4. Godfather, The (1972)
  5. Close Shave, A (1995)


--- Top User ---
User ID: 4064
Recommended Movies:
  1. Kids (1995)
  2. Choices (1981)
  3. Century (1993)
  4. Windows (1980)
  5. Love, etc. (1996)




In [75]:
BUCKET_NAME = 'yi-alex-hw2'
S3_EMBEDDINGS_KEY = 'bert_embeddings_fulldata.pkl'
EXTRACT_DIR = 'ml-1m-data'
OUTPUT_FILE_NAME = 'alex_recommendations.json'

my_ratings = {
    3000: 5, # Princess Mononoke
    1: 5, # Toy Story
    1210: 4, # Star Wars: Return of the Jedi
    2628: 4, # Star Wars: The Phantom Menace
    2080: 4, # Lady and the Tramp 
    318: 5, # Shawshank Redemption
    1196: 5, # Star Wars: The Empire Strikes Back
    260: 5, # Star Wars: A New Hope
    2018: 4, # Bambi
    3314: 5, # Toy Story 2
}

generate_recommendations(BUCKET_NAME, S3_EMBEDDINGS_KEY, EXTRACT_DIR, OUTPUT_FILE_NAME, True, my_ratings)

print_recommnedations('alex_recommendations.json')

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Loading BERT embeddings from bert_embeddings_fulldata.pkl...
Loading Ratings and Users datasets...
Generating Personal recommendations...
Uploading alex_recommendations.json to S3...
Recommendations saved to S3 as 'alex_recommendations.json'.
--- Personal Profile ---
Recommended Movies:
  1. Kids (1995)
  2. Lord of the Rings, The (1978)
  3. Space Jam (1996)
  4. Star Kid (1997)
  5. Swing Kids (1993)


